# Daily IC screen

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd() / 'scratchpad/hyperliquid-ic' if (Path.cwd() / 'scratchpad/hyperliquid-ic').exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / '_artifacts-rewrite'
ARTIFACTS.mkdir(exist_ok=True)
from ic_research import ResearchConfig, blockwise_prediction_permutation_null, completed_daily_window, compute_ic_table, daily_prediction_ic, feature_names, matched_daily_feature_control_ic, monthly_walk_forward_ridge, moving_block_bootstrap_ic, paired_block_bootstrap_difference, prediction_ic

CONFIG = ResearchConfig()
features = pd.read_parquet(ARTIFACTS / 'features.parquet')
labels = pd.read_parquet(ARTIFACTS / 'labels.parquet')
observations = pd.read_parquet(ARTIFACTS / 'observations.parquet')
evaluation_start, evaluation_end = completed_daily_window(observations)
short_core = feature_names(features, include_optional=False)
optional = feature_names(features, include_optional=True)
screen_features = [name for name in optional if not name.endswith('__missing')]
targets = [f'forward_{kind}_{horizon}' for horizon in CONFIG.forecast_horizons for kind in ('log_growth', 'downside_semivariance', 'variance')]
print('Evaluation:', evaluation_start.date(), 'to', evaluation_end.date(), '| daily decisions:', (evaluation_end - evaluation_start).days + 1, '| core:', len(short_core), '| IC features:', len(screen_features), '| optional model features:', len(optional))


Evaluation: 2025-09-13 to 2026-09-12 | daily decisions: 365 | core: 49 | IC features: 200 | optional model features: 401


In [2]:
ic = compute_ic_table(features, labels, feature_columns=screen_features, target_columns=targets, start=evaluation_start, end=evaluation_end)
ic.to_parquet(ARTIFACTS / 'ic-results.parquet', index=False)
display(ic.sort_values('mean_abs_rank_ic', ascending=False).head(30))


,feature,target,mean_rank_ic,median_rank_ic,mean_abs_rank_ic,ic_ir,dates
317,interval_vol_7,forward_variance_7,0.804325,0.826043,0.804325,9.678081,235
320,interval_vol_7,forward_variance_14,0.799375,0.818191,0.799375,10.648629,228
656,interval_vol_14,forward_variance_14,0.786645,0.813466,0.786645,8.877045,228
1139,daily_downside_dev_21,forward_variance_14,0.784676,0.796301,0.784676,16.451639,207
653,interval_vol_14,forward_variance_7,0.784407,0.809665,0.784407,9.696837,235
3365,btc_residual_vol_14,forward_variance_14,0.783196,0.791219,0.783196,7.756163,206
323,interval_vol_7,forward_variance_21,0.782436,0.803572,0.782436,8.788124,221
761,daily_downside_dev_14,forward_variance_14,0.781392,0.793311,0.781392,13.511495,212
1142,daily_downside_dev_21,forward_variance_21,0.780836,0.789887,0.780836,17.874569,200
1034,interval_vol_21,forward_variance_14,0.780471,0.799200,0.780471,9.568168,228


In [3]:
model_rows = []
for horizon in CONFIG.forecast_horizons:
    for kind in ('log_growth', 'downside_semivariance', 'variance'):
        target = f'forward_{kind}_{horizon}'
        model_rows.append(monthly_walk_forward_ridge(features, labels, feature_columns=short_core, target=target, horizon=horizon, evaluation_start=evaluation_start, evaluation_end=evaluation_end))
predictions = pd.concat(model_rows, ignore_index=True)
predictions.to_parquet(ARTIFACTS / 'predictions-short-core.parquet', index=False)
model_ic = prediction_ic(predictions)
model_ic.to_csv(ARTIFACTS / 'prediction-ic-short-core.csv', index=False)
display(model_ic)
print('Warm-up rows:', int((predictions.status == 'warmup').sum()), '| predicted rows:', int((predictions.status == 'predicted').sum()))


,target,horizon,mean_rank_ic,dates,ic_ir
0,forward_downside_semivariance_14,14,0.473757,151,3.175545
1,forward_downside_semivariance_21,21,0.532646,144,3.937710
2,forward_downside_semivariance_30,30,0.512683,105,3.673340
3,forward_downside_semivariance_45,45,0.398790,90,3.480894
4,forward_downside_semivariance_60,60,0.320654,75,2.822100
5,forward_downside_semivariance_7,7,0.505697,158,3.364430
6,forward_downside_semivariance_90,90,0.448083,14,12.425156
7,forward_log_growth_14,14,0.051152,351,0.282691
8,forward_log_growth_21,21,0.090974,344,0.584393
9,forward_log_growth_30,30,0.122402,335,0.909989


Warm-up rows: 3102 | predicted rows: 1152739


In [4]:
# Matched optional comparison is intentionally limited to forward growth.
optional_rows = []
for horizon in CONFIG.forecast_horizons:
    optional_rows.append(monthly_walk_forward_ridge(features, labels, feature_columns=optional, target=f'forward_log_growth_{horizon}', horizon=horizon, evaluation_start=evaluation_start, evaluation_end=evaluation_end))
optional_predictions = pd.concat(optional_rows, ignore_index=True)
optional_predictions.to_parquet(ARTIFACTS / 'predictions-optional-growth.parquet', index=False)
comparison = prediction_ic(optional_predictions)
comparison.to_csv(ARTIFACTS / 'prediction-ic-optional-growth.csv', index=False)
display(comparison)


,target,horizon,mean_rank_ic,dates,ic_ir
0,forward_log_growth_14,14,0.011339,351,0.076209
1,forward_log_growth_21,21,0.052307,344,0.410665
2,forward_log_growth_30,30,0.072173,335,0.722082
3,forward_log_growth_45,45,0.101537,320,1.058048
4,forward_log_growth_60,60,0.069375,305,0.713356
5,forward_log_growth_7,7,0.002327,358,0.016695
6,forward_log_growth_90,90,0.052166,275,0.398389


In [5]:
# Dependence-aware validation, matched controls and cohort reporting.
import numpy as np

daily_ic = daily_prediction_ic(predictions)
daily_ic.to_parquet(ARTIFACTS / 'daily-prediction-ic-short-core.parquet', index=False)
bootstrap_rows = []
control_rows = []
for horizon in CONFIG.forecast_horizons:
    target = f'forward_log_growth_{horizon}'
    model_daily = daily_ic[(daily_ic.target == target) & (daily_ic.horizon == horizon)]
    for block_length, variant in [(horizon, 'H'), (horizon + 14, 'H_plus_14')]:
        interval = moving_block_bootstrap_ic(model_daily, block_length=block_length, draws=1_000, seed=horizon + block_length)
        interval['block_variant'] = variant
        bootstrap_rows.append(interval)
    for control_name, feature, sign, zero_downside in [
        ('low_interval_downside_30', 'interval_downside_dev_30', -1.0, True),
        ('short_return_30', 'interval_log_growth_30', 1.0, False),
    ]:
        matched_model_daily, control_daily = matched_daily_feature_control_ic(
            predictions, features, feature=feature, target=target, sign=sign,
            zero_downside=zero_downside,
        )
        paired = paired_block_bootstrap_difference(
            matched_model_daily, control_daily, block_length=horizon + 14,
            draws=1_000, seed=horizon,
        )
        control_rows.append({'target': target, 'horizon': horizon, 'control': control_name, **paired})
bootstrap = pd.concat(bootstrap_rows, ignore_index=True)
bootstrap.to_csv(ARTIFACTS / 'prediction-ic-block-bootstrap.csv', index=False)
controls = pd.DataFrame(control_rows)
controls.to_csv(ARTIFACTS / 'growth-model-matched-controls.csv', index=False)

def cohort_ic(column: str) -> pd.DataFrame:
    rows = []
    for (target, horizon, cohort, date), group in growth_with_cohorts.groupby(['target', 'horizon', column, 'date'], observed=True):
        sample = group[['prediction', 'actual']].dropna()
        if len(sample) >= 10 and sample.prediction.nunique() > 1 and sample.actual.nunique() > 1:
            rows.append({'target': target, 'horizon': horizon, 'cohort_type': column, 'cohort': str(cohort), 'date': date, 'rank_ic': sample.prediction.corr(sample.actual, method='spearman'), 'vaults': len(sample)})
    daily = pd.DataFrame(rows)
    if daily.empty:
        return daily
    return daily.groupby(['target', 'horizon', 'cohort_type', 'cohort'], as_index=False).agg(
        mean_rank_ic=('rank_ic', 'mean'), dates=('date', 'nunique'),
        mean_vaults=('vaults', 'mean'),
    )

growth_with_cohorts = predictions[
    (predictions.status == 'predicted') & predictions.target.str.startswith('forward_log_growth_')
].merge(features[['date', 'address', 'available_history_days', 'last_interval_days']], on=['date', 'address'], how='left')
growth_with_cohorts['age_cohort'] = pd.cut(
    growth_with_cohorts.available_history_days,
    bins=[0, 7, 14, 30, 90, 180, 360, np.inf], right=False,
    labels=['<7', '7-13', '14-29', '30-89', '90-179', '180-359', '360+'],
)
growth_with_cohorts['sampling_cohort'] = np.select(
    [growth_with_cohorts.last_interval_days.le(1.5), growth_with_cohorts.last_interval_days.between(5.5, 8.5)],
    ['daily', 'weekly'], default='irregular',
)
cohorts = pd.concat([cohort_ic('age_cohort'), cohort_ic('sampling_cohort')], ignore_index=True)
cohorts.to_csv(ARTIFACTS / 'growth-prediction-cohorts.csv', index=False)

growth_predictions = predictions[predictions.target.str.startswith('forward_log_growth_')]
permutation_null = blockwise_prediction_permutation_null(growth_predictions, block_length=90, draws=200, seed=1)
permutation_null.to_parquet(ARTIFACTS / 'growth-prediction-permutation-null.parquet', index=False)
observed_growth = model_ic[model_ic.target.str.startswith('forward_log_growth_')]
observed_max = observed_growth.mean_rank_ic.abs().max()
null_exceedances = int((permutation_null.max_abs_mean_rank_ic >= observed_max).sum())
null_p = (1 + null_exceedances) / (1 + len(permutation_null))
null_summary = pd.DataFrame({'observed_max_abs_mean_rank_ic': [observed_max], 'null_draws': [len(permutation_null)], 'null_exceedances': [null_exceedances], 'block_length_days': [90], 'familywise_permutation_p': [null_p], 'scope': ['prediction-level null; model selection is not rerun under the null']})
null_summary.to_csv(ARTIFACTS / 'growth-prediction-permutation-summary.csv', index=False)
display(bootstrap[bootstrap.target.str.startswith('forward_log_growth_')])
display(controls)
display(cohorts[(cohorts.target == 'forward_log_growth_30')])
display(null_summary)


,target,horizon,block_length_days,dates,nominal_blocks,mean_rank_ic,bootstrap_p05,bootstrap_p50,bootstrap_p95,positive_date_fraction,block_variant
0,forward_log_growth_7,7,7,358,51.142857,0.038042,0.009642,0.038555,0.066361,0.603352,H
1,forward_log_growth_7,7,21,358,17.047619,0.038042,0.011585,0.038412,0.064807,0.603352,H_plus_14
2,forward_log_growth_14,14,14,351,25.071429,0.051152,0.012520,0.051731,0.096106,0.641026,H
3,forward_log_growth_14,14,28,351,12.535714,0.051152,0.012084,0.051252,0.091876,0.641026,H_plus_14
4,forward_log_growth_21,21,21,344,16.380952,0.090974,0.044913,0.091882,0.131722,0.720930,H
5,forward_log_growth_21,21,35,344,9.828571,0.090974,0.048362,0.090793,0.130813,0.720930,H_plus_14
6,forward_log_growth_30,30,30,335,11.166667,0.122402,0.088261,0.124407,0.156214,0.826866,H
7,forward_log_growth_30,30,44,335,7.613636,0.122402,0.090696,0.122910,0.149645,0.826866,H_plus_14
8,forward_log_growth_45,45,45,320,7.111111,0.160538,0.119195,0.160188,0.201976,0.884375,H
9,forward_log_growth_45,45,59,320,5.423729,0.160538,0.118632,0.159050,0.202564,0.884375,H_plus_14


,target,horizon,control,paired_dates,mean_difference,bootstrap_p05,bootstrap_p95
0,forward_log_growth_7,7,low_interval_downside_30,358,-0.023184,-0.047029,-0.002835
1,forward_log_growth_7,7,short_return_30,358,0.007194,-0.023507,0.037088
2,forward_log_growth_14,14,low_interval_downside_30,351,-0.046006,-0.080342,-0.016294
3,forward_log_growth_14,14,short_return_30,351,-0.007542,-0.056647,0.041344
4,forward_log_growth_21,21,low_interval_downside_30,344,-0.030968,-0.056328,-0.004943
5,forward_log_growth_21,21,short_return_30,344,0.023600,-0.024450,0.074316
6,forward_log_growth_30,30,low_interval_downside_30,335,-0.027252,-0.065489,0.005287
7,forward_log_growth_30,30,short_return_30,335,0.035248,-0.028552,0.093301
8,forward_log_growth_45,45,low_interval_downside_30,320,-0.013738,-0.043686,0.016533
9,forward_log_growth_45,45,short_return_30,320,0.060889,-0.006146,0.143164


,target,horizon,cohort_type,cohort,mean_rank_ic,dates,mean_vaults
14,forward_log_growth_30,30,age_cohort,14-29,0.161693,208,14.576923
15,forward_log_growth_30,30,age_cohort,180-359,0.116556,335,67.411940
16,forward_log_growth_30,30,age_cohort,30-89,0.180885,334,35.997006
17,forward_log_growth_30,30,age_cohort,360+,0.095963,335,58.683582
18,forward_log_growth_30,30,age_cohort,7-13,0.326421,7,10.857143
19,forward_log_growth_30,30,age_cohort,90-179,0.119023,335,54.182090
20,forward_log_growth_30,30,age_cohort,<7,0.338788,5,10.200000
53,forward_log_growth_30,30,sampling_cohort,daily,0.117922,335,219.131343
54,forward_log_growth_30,30,sampling_cohort,irregular,0.085022,13,56.769231
55,forward_log_growth_30,30,sampling_cohort,weekly,0.161970,139,25.553957


,observed_max_abs_mean_rank_ic,null_draws,null_exceedances,block_length_days,familywise_permutation_p,scope
0,0.160538,200,0,90,0.004975,prediction-level null; model selection is not ...
